# Vectrix: Model Comparison & DNA Deep Dive

**Explore 30+ forecasting models and understand your data's DNA fingerprint.**

| What you'll learn | Time |
|:------------------|:-----|
| Compare all models on your data | 2 min |
| Understand DNA profiling (65+ features) | 2 min |
| Select specific models | 1 min |
| Control ensemble strategy | 1 min |
| Adaptive forecasting (regime, constraints) | 2 min |

---

In [ ]:
!pip install -q vectrix

## 1. Full Model Comparison

Run every available model on the same dataset and rank by accuracy.

In [ ]:
from vectrix import forecast, compare, analyze, loadSample

df = loadSample("airline")
comp = compare(df, date="date", value="passengers", steps=12)
print(comp.summary())

In [ ]:
# Full comparison table with all metrics
comp.table()

In [ ]:
# All model forecasts as a DataFrame
all_fc = comp.all_forecasts()
print(f"Models that produced valid forecasts: {len(all_fc.columns) - 1}")
all_fc.head()

## 2. DNA Profiling — 65+ Features

Every time series has a unique DNA fingerprint. Vectrix extracts 65+ statistical features that characterize your data.

In [ ]:
analysis = analyze(df, date="date", value="passengers")
dna = analysis.dna

print("=== Time Series DNA ===")
print(f"Trend strength:        {dna.trendStrength:.3f}")
print(f"Seasonal strength:     {dna.seasonalStrength:.3f}")
print(f"Noise level:           {dna.noiseLevel:.3f}")
print(f"Difficulty:            {dna.difficulty}")
print(f"Stationarity (ADF):    {dna.isStationary}")
print(f"Hurst exponent:        {dna.hurstExponent:.3f}")
print(f"Dominant period:       {dna.seasonalPeakPeriod}")

In [ ]:
# DNA-recommended models (ranked by expected fit)
print("\nRecommended models (DNA-based):")
for i, model in enumerate(dna.recommendedModels[:5], 1):
    print(f"  {i}. {model}")

In [ ]:
# Changepoint detection
print(f"\nChangepoints detected: {len(analysis.changepoints)}")
for cp in analysis.changepoints[:5]:
    print(f"  Index {cp['index']}: magnitude={cp.get('magnitude', 'N/A')}")

## 3. Level 2: Select Specific Models

Override automatic model selection and choose exactly which models to run.

In [ ]:
# Use only DOT (the strongest single model)
result_dot = forecast(df, date="date", value="passengers", steps=12, models=["dot"])
print(f"DOT only — MAPE: {result_dot.mape:.2f}%")

# Use only AutoCES
result_ces = forecast(df, date="date", value="passengers", steps=12, models=["auto_ces"])
print(f"AutoCES only — MAPE: {result_ces.mape:.2f}%")

# Use the Core3 ensemble (DOT + CES + 4Theta)
result_core3 = forecast(df, date="date", value="passengers", steps=12, models=["dot", "auto_ces", "four_theta"])
print(f"Core3 ensemble — MAPE: {result_core3.mape:.2f}%")

In [ ]:
# Control ensemble strategy
result_mean = forecast(df, date="date", value="passengers", steps=12, ensemble="mean")
result_median = forecast(df, date="date", value="passengers", steps=12, ensemble="median")
result_best = forecast(df, date="date", value="passengers", steps=12, ensemble="best")

print(f"Mean ensemble   — MAPE: {result_mean.mape:.2f}%")
print(f"Median ensemble — MAPE: {result_median.mape:.2f}%")
print(f"Best single     — MAPE: {result_best.mape:.2f}%")

## 4. Different Data Types

See how DNA and model selection change with data characteristics.

In [ ]:
# Stock price — random walk, no seasonality
stock = loadSample("stock")
stock_analysis = analyze(stock, date="date", value="close")
stock_dna = stock_analysis.dna

print("=== Stock Price DNA ===")
print(f"Trend:     {stock_dna.trendStrength:.3f}")
print(f"Seasonal:  {stock_dna.seasonalStrength:.3f}")
print(f"Noise:     {stock_dna.noiseLevel:.3f}")
print(f"Difficulty: {stock_dna.difficulty}")
print(f"Top model:  {stock_dna.recommendedModels[0]}")

In [ ]:
# Intermittent demand — sparse, lumpy
intermittent = loadSample("intermittent")
int_analysis = analyze(intermittent, date="date", value="demand")
int_dna = int_analysis.dna

print("=== Intermittent Demand DNA ===")
print(f"Trend:     {int_dna.trendStrength:.3f}")
print(f"Seasonal:  {int_dna.seasonalStrength:.3f}")
print(f"Noise:     {int_dna.noiseLevel:.3f}")
print(f"Difficulty: {int_dna.difficulty}")
print(f"Top model:  {int_dna.recommendedModels[0]}")

In [ ]:
# Energy (hourly) — multi-seasonal
energy = loadSample("energy")
energy_analysis = analyze(energy, date="date", value="consumption_kwh")
energy_dna = energy_analysis.dna

print("=== Energy Consumption DNA ===")
print(f"Trend:     {energy_dna.trendStrength:.3f}")
print(f"Seasonal:  {energy_dna.seasonalStrength:.3f}")
print(f"Noise:     {energy_dna.noiseLevel:.3f}")
print(f"Difficulty: {energy_dna.difficulty}")
print(f"Top model:  {energy_dna.recommendedModels[0]}")

## 5. Adaptive Intelligence

Vectrix includes adaptive features: regime detection, self-healing forecasts, and constraint-aware predictions.

In [ ]:
from vectrix import Vectrix
import numpy as np

vx = Vectrix(df, dateCol="date", valueCol="passengers")
vx.fit()

# Regime detection — find structural shifts in the data
regimes = vx.detectRegimes()
print(f"Number of regimes: {len(regimes.regimeStats)}")
for i, stat in enumerate(regimes.regimeStats):
    print(f"  Regime {i}: mean={stat['mean']:.1f}, std={stat['std']:.1f}, size={stat['size']}")

In [ ]:
# Constrained forecasting — ensure predictions respect business rules
result_constrained = vx.forecast(
    steps=12,
    constraints=[{"type": "min", "value": 0}]  # No negative passengers
)
print("Constrained forecast (min=0):")
print(f"  Min predicted: {min(result_constrained.forecast):.1f}")
print(f"  Max predicted: {max(result_constrained.forecast):.1f}")

---

## What's Next?

- **[Try with Your Data](https://colab.research.google.com/github/eddmpython/vectrix/blob/master/notebooks/03_try_your_data.ipynb)** — Upload your CSV
- **[Full Documentation](https://eddmpython.github.io/vectrix/docs/)** — Guides, API reference, tutorials
- **[Benchmarks](https://eddmpython.github.io/vectrix/docs/benchmarks/)** — M4 Competition results (OWA 0.877)